In [ ]:
# Header for the notebook
from datetime import datetime
from IPython.display import display, Markdown

# Get the current date
title = "ENMO_00.jpynb"
current_date = datetime.now().strftime("%d %B %Y, %H:%M:%S")
url_repository = "https://github.com/MarwaSakh/HAH913E-Physical-activity-00-Assignment.git"
authors = "Marwa Sakher, Hafsa Raissi, Jérôme Bosc and their AI friends"

# Insert the date into the notebook
display(Markdown(f"# {title}"))
display(Markdown(f"{current_date}"))
display(Markdown(f"Repository URL: <{url_repository}>"))
display(Markdown(f"Team members: {authors}"))

## Objective

We have a few minutes of raw data from an accelerometer worn on the non-dominant wrist (Axivity AX3).
The file `0_z.csv` contains 4 columns: `t` (seconds), `x`, `y`, `z` (in g).

The goal is to calculate the **ENMO** (*Euclidean Norm Minus One*), a measure of physical activity intensity,
then **integrate** it over 10-second, 30-second, and 60-second intervals, and plot the result.

## Imports and Parameters

The file begins with a comment line (`# accelerometer data in g`): this line is skipped during reading.
The requested epochs are grouped into a list so they can be looped through.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = "../data/0_z.csv"
EPOCH_LENGTHS = [10, 30, 60]

## Loading Data

`load_data` reads the CSV file and returns a DataFrame. `get_sampling_rate` calculates the **sampling rate**
based on the median difference between two timestamps (expected: 0.02 s, or 50 Hz).

In [ ]:
def load_data(path):
    """Load data (colonnes t, x, y, z), ignore the comment line."""
    return pd.read_csv(path, comment="#")

df = load_data(CSV_PATH)
df.head()

## Calculating ENMO

Prompt to AI: How calculate ENMO with Python ?  

AI gave us the formula to calculate it, and it suggested removing the negative part.

At rest, an accelerometer measures only gravity: the magnitude of the vector (x, y, z) is approximately **1 g**.
For each data point: $ENMO = \sqrt{x^2 + y^2 + z^2} - 1$ (in g).

**Handling negative values.** In the literature, ENMO is often clipped to 0:
$\max(0,\ \sqrt{x^2+y^2+z^2}-1)$, because a negative value has no physical meaning (it is noise or a slight
calibration error: here, the norm at rest is approximately 0.955 g). Here, we **retain negative values by default**
(`clip_negatives=False`), as in the example provided by the instructor, which results in very small negative values
at rest. The parameter `clip_negatives=True` allows you to switch to the “standard” version in a single line.

In [ ]:
def compute_enmo(df, clip_negatives=False):
    """ENMO (en g) pour chaque échantillon : norme(x, y, z) - 1.
    Si clip_negatives=True, les valeurs négatives sont mises à 0."""
    enmo = np.sqrt(df["x"]**2 + df["y"]**2 + df["z"]**2) - 1.0
    return enmo.clip(lower=0.0) if clip_negatives else enmo


df["enmo"] = compute_enmo(df)

In [ ]:
def integrate_enmo(df, epoch_s):
    """
    Intègre l'ENMO sur des époques de epoch_s secondes.
    Résultat en g·min : somme(enmo * dt) / 60.
    """
    dt = df["t"].diff().median()               # pas d'échantillonnage (s)
    t0 = df["t"].iloc[0]
    epoch_idx = ((df["t"] - t0) // epoch_s).astype(int)

    integrated = (df["enmo"] * dt).groupby(epoch_idx).sum() / 60.0
    epochs_df = pd.DataFrame({
        "epoch_start": integrated.index * epoch_s,
        "enmo_integre": integrated.values,
    })
    return epochs_df

In [ ]:
def plot_enmo(df, epochs_df, epoch_s):
    """Figure à 2 graphiques : ENMO brut, puis ENMO intégré par époque."""
    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    axes[0].plot(df["t"], df["enmo"], lw=0.5)
    axes[0].set_ylabel("ENMO (g)")
    axes[0].set_title(f"ENMO brut")

    axes[1].bar(epochs_df["epoch_start"], epochs_df["enmo_integre"],
                width=epoch_s, align="edge", edgecolor="k")
    axes[1].set_ylabel("ENMO intégré (g·min)")
    axes[1].set_xlabel("Temps (s)")
    axes[1].set_title(f"ENMO intégré sur des époques de {epoch_s} s")

    plt.tight_layout()
    plt.show()


def run_all(path, epoch_lengths=(10, 30, 60)):
    df = compute_enmo(load_data(path))
    for epoch_s in epoch_lengths:
        epochs_df = integrate_enmo(df, epoch_s)
        plot_enmo(df, epochs_df, epoch_s)


run_all("../data/0_z.csv")

In [ ]:
import os
print("Current directory:", os.getcwd())
print("Contents:", os.listdir("."))